# Validazione Finale Embedding (42k Dataset)
Questo notebook analizza la qualità geometrica e la stabilità dei vettori definitivi generati sul dataset completo da 42.000 email, utilizzando il modello `bge-small` ottimizzato.


In [4]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.embedding_comparision import (

    cluster_kmeans,
    compute_clustering_metrics,
    compute_stability_metrics,
    compute_retrieval_metrics
)

ENV_PATH = PROJECT_ROOT / ".env"
ENV = dotenv_values(ENV_PATH)

EMBEDDINGS_PATH = PROJECT_ROOT / "data/embeddings/email_embeddings.npy"
INDEX_PATH = PROJECT_ROOT / "data/metadata/email_embedding_index.parquet"
METADATA_PATH = PROJECT_ROOT / "data/metadata/email_embedding_metadata.json"



## 1. Caricamento Dati


In [5]:
if not EMBEDDINGS_PATH.exists():
    print("In attesa che la pipeline di embedding finisca...")
else:
    embeddings = np.load(EMBEDDINGS_PATH)
    index_df = pd.read_parquet(INDEX_PATH)
    with METADATA_PATH.open(encoding="utf-8") as f:
        metadata = json.load(f)
    print(f"Caricati {embeddings.shape[0]} vettori di dimensione {embeddings.shape[1]}")
    print(f"Modello originale: {metadata.get("model_name")}")



Caricati 1757624 vettori di dimensione 384
Modello originale: BAAI/bge-small-en-v1.5


## 2. Sweep di K (Alla ricerca dei Macro-Cluster)
Visto che abbiamo 42.000 email, testeremo un numero di cluster K più alto (es. 20, 30, 40, 50, 60) per trovare la suddivisione geometricamente più stabile.


In [6]:
K_VALUES = [20, 30, 40, 50, 60]
clustering_results = []

if EMBEDDINGS_PATH.exists():
    for k in K_VALUES:
        print(f"\nAnalizzando K={k}...")
        labels, _ = cluster_kmeans(embeddings, n_clusters=k)
        metrics = compute_clustering_metrics(embeddings, labels)
        
        clustering_results.append({
            "k": k,
            "silhouette": metrics["silhouette"],
            "davies_bouldin": metrics["davies_bouldin"],
            "calinski_harabasz": metrics["calinski_harabasz"],
        })
        print(f"Silhouette: {metrics["silhouette"]:.4f} | DB: {metrics["davies_bouldin"]:.4f} | CH: {metrics["calinski_harabasz"]:.1f}")

    df_metrics = pd.DataFrame(clustering_results)
    display(df_metrics)
else:
    print("Dati non ancora pronti.")




Analizzando K=20...
Dataset molto grande (1757624 samples). Uso MiniBatchKMeans per ottimizzazione.
Silhouette: 0.0215 | DB: 4.2581 | CH: 269.0

Analizzando K=30...
Dataset molto grande (1757624 samples). Uso MiniBatchKMeans per ottimizzazione.
Silhouette: 0.0121 | DB: 4.2505 | CH: 201.3

Analizzando K=40...
Dataset molto grande (1757624 samples). Uso MiniBatchKMeans per ottimizzazione.
Silhouette: 0.0136 | DB: 4.1581 | CH: 163.4

Analizzando K=50...
Dataset molto grande (1757624 samples). Uso MiniBatchKMeans per ottimizzazione.
Silhouette: 0.0130 | DB: 4.2131 | CH: 139.2

Analizzando K=60...
Dataset molto grande (1757624 samples). Uso MiniBatchKMeans per ottimizzazione.
Silhouette: 0.0125 | DB: 4.0985 | CH: 122.4


,k,silhouette,davies_bouldin,calinski_harabasz
0,20,0.021492,4.258051,269.013651
1,30,0.012108,4.250508,201.336264
2,40,0.013582,4.158094,163.388994
3,50,0.013032,4.213076,139.234211
4,60,0.012466,4.098493,122.408876


## 3. Stabilità Bootstrap sul K Migliore\nCalcoliamo la stabilità reale simulando variazioni sul dataset.


In [7]:
if EMBEDDINGS_PATH.exists() and len(clustering_results) > 0:
    best_k_row = df_metrics.sort_values("silhouette", ascending=False).iloc[0]
    best_k = int(best_k_row["k"])
    print(f"Eseguo bootstrap stability su K={best_k} (miglior Silhouette)...")
    
    # Ricalcolo labels per K migliore
    labels, _ = cluster_kmeans(embeddings, n_clusters=best_k)
    
    # Bootstrap (riduciamo iterazioni a 5 per questioni di tempo visto il dataset enorme)
    stability = compute_stability_metrics(embeddings, labels, bootstrap_iterations=5, n_clusters=best_k)
    
    print(f"\nStabilità K={best_k}:")
    print(f"ARI (mean): {stability["ari_mean"]:.4f} ± {stability["ari_std"]:.4f}")
    print(f"NMI (mean): {stability["nmi_mean"]:.4f} ± {stability["nmi_std"]:.4f}")



Eseguo bootstrap stability su K=20 (miglior Silhouette)...
Dataset molto grande (1757624 samples). Uso MiniBatchKMeans per ottimizzazione.
Dataset molto grande (1230336 samples). Uso MiniBatchKMeans per ottimizzazione.
Dataset molto grande (1230336 samples). Uso MiniBatchKMeans per ottimizzazione.
Dataset molto grande (1230336 samples). Uso MiniBatchKMeans per ottimizzazione.
Dataset molto grande (1230336 samples). Uso MiniBatchKMeans per ottimizzazione.
Dataset molto grande (1230336 samples). Uso MiniBatchKMeans per ottimizzazione.

Stabilità K=20:
ARI (mean): 0.4843 ± 0.0411
NMI (mean): 0.6390 ± 0.0258
